# 최종 제출 노트북: 22b XGB 시드 평균 0.7 + LR 0.3 (LB 0.3792)

이 노트북 하나만 위에서부터 끝까지 실행하면 최종 제출 파일이 만들어집니다. 외부 모듈(`src/`)을 불러오지 않고, 데이터 전처리·피처·검증·학습·저장 코드를 모두 이 안에 담았습니다.

| 단계 | 내용 |
| :--- | :--- |
| 1. 전처리 | 빈칸 → `WT`, 변이 표기 통일 (HGVS 표준: `X` → `*`, frameshift 뒤 아미노산 제거) |
| 2. 피처 | 유전자 변이 유무(0/1) + 유형별 변이 비율 + 전체 변이 수 log + 미스센스 치환 종류 비율 |
| 3. 모델 | XGBoost(시드 5개 평균) × 0.7 + 로지스틱 회귀 × 0.3 |
| 4. 검증 | Stratified 5-Fold (seed 42), Macro F1 |
| 5. 저장 | `submissions/날짜_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_OOF.csv` |

**실행 방법**
- 폴더 구조: 이 노트북은 `notebooks/`에, 데이터는 `data/train.csv`, `data/test.csv`, `data/sample_submission.csv`에 둡니다 (다른 곳이면 아래 `DATA_DIR`만 바꾸면 됨)
- 패키지: `requirements.txt` (Python 3.12, xgboost 3.x, scikit-learn, pandas, numpy)
- 실행 시간: 약 20분 (XGB 시드 5개 × 5 fold + LR)
- 최종 LB 0.3792 파일은 **Windows 데스크탑**에서 만들었습니다. XGBoost는 CPU 종류(Windows x86 / macOS M1)에 따라 확률이 아주 조금 달라서, 다른 컴퓨터에서는 CV가 ±0.004 정도 다를 수 있습니다 (같은 컴퓨터에서는 항상 같은 결과).

**Data Leakage를 막기 위한 원칙**
- test는 전처리·피처를 **같은 규칙으로 변환**하는 데만 쓰고, test에서 계산한 통계나 test 분포를 보고 정한 기준은 쓰지 않습니다.
  - 그래서 test 패턴을 보고 만들었던 **이소폼 중복 제거는 끄고**, test 개수를 보고 정했던 **in-frame indel 제외도 되돌려 모든 변이 유형을 포함**합니다 (11/13, 운영진 데이터 누수 공지 반영).
  - 표기 통일은 test 분포가 아니라 외부 표준(HGVS 표기법)에 맞춘 형식 변환이라 유지합니다.
  - 이 전처리·피처는 #43 머지 후 `src/data.py`의 `load_data()` 기본값(`dedupe=False`)과 `src/features.py`의 `mutation_features()` 결과와 똑같습니다 (11/13 대조 확인).
- 피처는 모두 **환자 한 명 안에서만** 계산합니다. 여러 환자의 통계가 필요한 단계(치환 종류 상위 100개 선택)는 **fold 학습 부분에서만** 정합니다.

## 0. 설정

In [1]:
import os, re, random, platform, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

# 경로: notebooks/에서 실행하면 ../data, 저장소 루트에서 실행하면 ./data
ROOT_DIR = Path('..') if Path('../data/train.csv').exists() else Path('.')
DATA_DIR = ROOT_DIR / 'data'
SUBMISSION_DIR = ROOT_DIR / 'submissions'

ID_COL, TARGET_COL, WT = 'ID', 'SUBCLASS', 'WT'
SEED, N_FOLDS = 42, 5
XGB_SEEDS = [42, 7, 123, 2024, 777]   # XGB 시드 평균에 쓰는 시드 5개 (최종 제출과 동일)
N_SUB = 100                            # 미스센스 치환 종류 상위 몇 개를 쓸지 (fold 학습 부분에서 선택)
W_XGB = 0.7                            # 섞는 비율: XGB 0.7 + LR 0.3

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
print(f'{platform.system()} | CPU {platform.machine()} | python {platform.python_version()} | '
      f'xgboost {xgb.__version__} | scikit-learn {sklearn.__version__} | pandas {pd.__version__}')
print('데이터 폴더:', DATA_DIR.resolve())

Darwin | CPU arm64 | python 3.12.14 | xgboost 3.4.1 | scikit-learn 1.9.1 | pandas 3.1.1
데이터 폴더: /Users/mac/web/dacon-cancer-subclass/data


## 1. 데이터 로드와 전처리
- `keep_default_na=False`: `NA` 같은 문자열이 결측으로 바뀌지 않게 읽습니다 (빈칸은 `''`로 읽힘)
- 빈칸 → `WT` (test에만 237칸)
- 변이 표기 통일 (train과 test에 같은 규칙 적용, HGVS 표준 표기법 기준)
  - 정지코돈 `X` → `*`: `R492X` → `R492*`, `X541X` → `*541*` (`X`는 예전 표기로 `*`와 같은 뜻, train에는 `X`가 없음)
  - frameshift 뒤 아미노산 제거: `K600Nfs` → `K600fs`
- 이소폼 중복 제거는 **하지 않습니다** (test 패턴을 보고 만든 규칙이라 최종 제출에서 제외)

In [2]:
_FS_AA = re.compile(r'^([A-Z*]*\d+)[A-Z]?fs.*$')   # K600Nfs, K600Nfs*12

def normalize_token(token):
    # 변이 토큰 1개의 표기 통일: X → * (토큰 안 모든 X), frameshift 뒤 아미노산 제거
    token = token.replace('X', '*')
    return _FS_AA.sub(r'\1fs', token)

def normalize_cell(cell):
    # 한 칸(공백으로 구분된 여러 토큰)의 모든 토큰 표기 통일, WT는 그대로
    if not isinstance(cell, str) or cell == WT:
        return cell
    return ' '.join(normalize_token(t) for t in cell.split())

read = dict(dtype=str, keep_default_na=False)
train = pd.read_csv(DATA_DIR / 'train.csv', **read)
test = pd.read_csv(DATA_DIR / 'test.csv', **read)
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

genes = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]
assert list(test.columns) == [ID_COL] + genes, 'train/test 유전자 컬럼이 다릅니다.'
for df in (train, test):
    df[genes] = df[genes].replace('', WT)
    df[genes] = df[genes].apply(lambda col: col.map(normalize_cell))

print(f'train {train.shape}, test {test.shape}, 유전자 {len(genes)}개, 암종 {train[TARGET_COL].nunique()}개')
print('test에 남은 X 표기:', int(test[genes].apply(lambda c: c.str.contains('X')).to_numpy().sum()))

train (6201, 4386), test (2546, 4385), 유전자 4384개, 암종 26개
test에 남은 X 표기: 0


## 2. 피처
모두 **환자 한 명 안에서만** 계산합니다 (test에 같은 함수를 적용해도 Data Leakage 아님).

| 이름 | 열 수 | 내용 |
| :--- | :-: | :--- |
| `any` | 4,384 | 유전자마다 변이가 있으면 1, 없으면 0 (침묵 변이 포함) |
| `ratio` | 8 | 변이 유형 7가지(침묵·미스센스·넌센스·프레임시프트·in-frame indel·stop_loss·기타)의 비율 + 전체 변이 수 log1p |
| 치환 비율 | train에 나온 종류 수 | 미스센스 변이 중 치환 종류(R→H, E→K 등)의 비율 → 모델에는 fold 학습 부분에서 고른 상위 100개만 들어감 |

- 비율 피처는 test 환자의 변이 수가 train의 약 2배여도 같은 암이면 값이 비슷하게 나오도록 고른 것입니다.
- 치환 종류는 암의 원인(흡연, 자외선, DNA 복구 고장 등)마다 잘 생기는 종류가 달라 암종 구분에 도움이 됩니다.

In [3]:
MUT_TYPES = ['silent', 'missense', 'nonsense', 'frameshift', 'inframe_indel', 'stop_loss', 'other']
_SNV = re.compile(r'([A-Z*])(\d+)([A-Z*])')   # G12D, R895R, R213*, *261*

def mutation_type(token):
    # 변이 토큰 1개 → 유형 (01 EDA와 같은 규칙)
    if 'fs' in token:
        return 'frameshift'
    if 'del' in token or 'ins' in token:
        return 'inframe_indel'
    if '>' in token:                       # 연속 아미노산 치환 (예: 1529_1530AL>AL)
        before, after = token.split('>', 1)
        ref = before.lstrip('0123456789_')
        if ref == after:
            return 'silent'
        if '*' in after and '*' not in ref:
            return 'nonsense'
        return 'missense'
    m = _SNV.fullmatch(token)
    if m:
        ref, alt = m.group(1), m.group(3)
        if ref == alt:
            return 'silent'
        if alt in ('*', 'X'):
            return 'nonsense'
        if ref in ('*', 'X'):
            return 'stop_loss'
        return 'missense'
    return 'other'

def aa_substitution(token):
    # 단일 아미노산 치환 토큰 → '원래>바뀐' (R132H → R>H), 그 외(frameshift, indel, 여러 아미노산 치환)는 None
    m = _SNV.fullmatch(token.replace('X', '*'))
    return f'{m.group(1)}>{m.group(3)}' if m else None

def any_features(df):
    # 유전자별 변이 유무 (WT → 0, 그 외 → 1)
    return (df[genes] != WT).astype(np.int8)

def ratio_features(df):
    # 유형별 변이 비율 (모든 유형 포함) + 전체 변이 수 log1p, 변이 0개 환자는 비율 모두 0
    V = df[genes].to_numpy()
    counts = np.zeros((len(df), len(MUT_TYPES)), np.float32)
    type_idx = {t: k for k, t in enumerate(MUT_TYPES)}
    rows, cols = np.nonzero(V != WT)
    for i, j in zip(rows, cols):
        for token in V[i, j].split():
            counts[i, type_idx[mutation_type(token)]] += 1
    total = counts.sum(axis=1, keepdims=True)
    ratio = pd.DataFrame(counts / np.maximum(total, 1), index=df.index, columns=[f'r_{t}' for t in MUT_TYPES])
    ratio['n_mut_log'] = np.log1p(total[:, 0])
    return ratio.astype(np.float32)

def sub_ratio_features(df):
    # 환자별 미스센스 치환 종류 비율 (여러 아미노산 치환 토큰은 단일 치환이 아니라 제외)
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]   # R>H → sub__R_H
    return M

t = time.time()
A_tr, A_te = any_features(train), any_features(test)
R_tr, R_te = ratio_features(train), ratio_features(test)
S_tr = sub_ratio_features(train)
S_te = sub_ratio_features(test).reindex(columns=S_tr.columns, fill_value=0)   # 열은 train 기준 (test에만 있는 종류는 버림)

X_xgb = pd.concat([A_tr, R_tr, S_tr], axis=1)        # XGB 입력: any + ratio + 치환 비율
X_xgb_te = pd.concat([A_te, R_te, S_te], axis=1)
X_lr, X_lr_te = A_tr.astype('float32'), A_te.astype('float32')   # LR 입력: any
print(f'피처 계산 {time.time() - t:.0f}s | XGB 입력 {X_xgb.shape[1]}열 (any {A_tr.shape[1]} + ratio {R_tr.shape[1]} + 치환 {S_tr.shape[1]}) | LR 입력 {X_lr.shape[1]}열')
print('ratio 열:', list(R_tr.columns))

피처 계산 6s | XGB 입력 4633열 (any 4384 + ratio 8 + 치환 241) | LR 입력 4384열
ratio 열: ['r_silent', 'r_missense', 'r_nonsense', 'r_frameshift', 'r_inframe_indel', 'r_stop_loss', 'r_other', 'n_mut_log']


## 3. 검증 도구와 모델
- 라벨: `LabelEncoder`를 train 라벨로만 fit (26개 암종 → 0~25)
- 검증: Stratified 5-Fold (shuffle, seed 42), 지표는 대회 평가 지표인 Macro F1
- `TopNSubs`: fold 학습 부분에서 비율 합이 큰 치환 종류 상위 100개만 남김 (Pipeline 안에서 fit → 검증·test는 transform만)
- `BalancedXGB`: 암종별 환자 수 차이를 `balanced` 샘플 가중치로 보정한 XGBoost (깊이 3, 학습률 0.1, 200그루, 열 샘플링 0.3)
- 로지스틱 회귀: `class_weight='balanced'`, C=0.1

In [4]:
le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL])
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

def run_cv(make_model, X, y, X_test):
    # fold마다 새 모델을 학습해 OOF 확률과 test 확률(fold 평균)을 돌려줌
    oof = np.zeros((len(y), len(le.classes_)))
    test_proba = np.zeros((len(X_test), len(le.classes_)))
    for tr_idx, va_idx in folds:
        model = make_model()
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])
        test_proba += model.predict_proba(X_test) / len(folds)
    return oof, test_proba

def fold_scores(oof):
    return np.array([macro_f1(y[va], oof[va].argmax(1)) for _, va in folds])

class TopNSubs(BaseEstimator, TransformerMixin):
    # fold 학습 부분에서만: 비율 합이 큰 치환 종류 상위 n개만 남기고 나머지 치환 열은 버림
    def __init__(self, n=N_SUB):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

class BalancedXGB(xgb.XGBClassifier):
    # 암종별 환자 수 차이 보정: balanced 샘플 가중치로 학습
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

XGB_PARAMS = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
                  tree_method='hist', eval_metric='mlogloss', n_jobs=-1)
print('fold별 검증 환자 수:', [len(va) for _, va in folds])

fold별 검증 환자 수: [1241, 1240, 1240, 1240, 1240]


## 4. 학습
- 로지스틱 회귀 1개 (무작위성 없음)
- XGBoost: 같은 fold에서 시드만 5개(42, 7, 123, 2024, 777)로 바꿔 학습하고 확률을 평균 → 시드 하나의 운에 덜 좌우됨

In [5]:
t = time.time()
lr_oof, lr_te = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
                       X_lr, y, X_lr_te)
print(f'LR: CV {fold_scores(lr_oof).mean():.4f} ({time.time() - t:.0f}s)')

xgb_oofs, xgb_tes = [], []
for s in XGB_SEEDS:
    t = time.time()
    params = dict(XGB_PARAMS, random_state=s)
    oof, te = run_cv(lambda: make_pipeline(TopNSubs(N_SUB), BalancedXGB(**params)), X_xgb, y, X_xgb_te)
    xgb_oofs.append(oof); xgb_tes.append(te)
    print(f'XGB 시드 {s}: CV {fold_scores(oof).mean():.4f} ({time.time() - t:.0f}s)')

LR: CV 0.3818 (3s)
XGB 시드 42: CV 0.4552 (202s)
XGB 시드 7: CV 0.4497 (197s)
XGB 시드 123: CV 0.4540 (190s)
XGB 시드 2024: CV 0.4530 (190s)
XGB 시드 777: CV 0.4526 (191s)


## 5. 섞기와 검증 점수
- 최종 확률 = XGB(시드 5개 평균) × 0.7 + LR × 0.3 (비율은 14번 실험에서 정한 값을 그대로 사용, 다시 고르지 않음)

In [6]:
xgb_oof, xgb_te = np.mean(xgb_oofs, axis=0), np.mean(xgb_tes, axis=0)
final_oof = W_XGB * xgb_oof + (1 - W_XGB) * lr_oof
final_te = W_XGB * xgb_te + (1 - W_XGB) * lr_te

rows = []
for name, oof in [('LR (any)', lr_oof), ('XGB 시드 5개 평균', xgb_oof), ('최종: XGB 0.7 + LR 0.3', final_oof)]:
    fs = fold_scores(oof)
    rows.append({'모델': name, 'CV (fold 평균)': round(fs.mean(), 4), 'std': round(fs.std(), 4),
                 'OOF Macro F1': round(macro_f1(y, oof.argmax(1)), 4), **{f'fold {i}': round(v, 4) for i, v in enumerate(fs)}})
pd.DataFrame(rows)

,모델,CV (fold 평균),std,OOF Macro F1,fold 0,fold 1,fold 2,fold 3,fold 4
0,LR (any),0.3818,0.0065,0.3830,0.3872,0.3733,0.3831,0.3755,0.3899
1,XGB 시드 5개 평균,0.4588,0.0105,0.4600,0.4484,0.4570,0.4747,0.4474,0.4664
2,최종: XGB 0.7 + LR 0.3,0.4648,0.0113,0.4659,0.4576,0.4479,0.4803,0.4657,0.4724


## 6. 제출 파일 저장
- 파일명: `날짜_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_OOF점수.csv` (팀 규칙 형식)
- 형식 확인: 행 수·ID 순서가 `sample_submission.csv`와 같은지, 예측 라벨이 train 암종 안에 있는지

In [7]:
sub = submission.copy()
sub[TARGET_COL] = le.inverse_transform(final_te.argmax(1))
assert len(sub) == len(test) and (sub[ID_COL].values == test[ID_COL].values).all(), 'ID 순서가 test와 다릅니다.'
assert set(sub[TARGET_COL]) <= set(le.classes_)

SUBMISSION_DIR.mkdir(exist_ok=True)
oof_score = macro_f1(y, final_oof.argmax(1))
path = SUBMISSION_DIR / f"{datetime.now():%m%d}_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_{oof_score:.3f}.csv"
sub.to_csv(path, index=False, encoding='utf-8-sig')
print('저장 완료:', path.resolve())
print('test 예측 암종 상위 5개:', sub[TARGET_COL].value_counts(normalize=True).head(5).round(3).to_dict())

저장 완료: /Users/mac/web/dacon-cancer-subclass/submissions/1113_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_0.466.csv
test 예측 암종 상위 5개: {'COAD': 0.181, 'STES': 0.179, 'KIPAN': 0.058, 'BRCA': 0.054, 'OV': 0.047}


## 7. (선택) 최종 제출 파일과 비교
- Windows 데스크탑에서 만든 최종 파일(LB 0.3792)이 `submissions/`에 있으면 예측이 얼마나 같은지 확인합니다.
- 같은 Windows 환경이면 100% 같고, 다른 CPU(macOS M1 등)에서는 XGBoost 확률 차이로 일부 환자가 다를 수 있습니다.

In [8]:
final_file = SUBMISSION_DIR / '1113_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_0.465.csv'
if final_file.exists():
    ref = pd.read_csv(final_file)
    print(f'최종 제출 파일(LB 0.3792)과 예측 일치율: {(ref[TARGET_COL].values == sub[TARGET_COL].values).mean():.1%}')
else:
    print('최종 제출 파일이 이 컴퓨터에 없어 비교를 건너뜀:', final_file.name)

최종 제출 파일이 이 컴퓨터에 없어 비교를 건너뜀: 1113_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_0.465.csv


## 결과 기록

| 실행 환경 | CV (fold 평균) | OOF Macro F1 | LB |
| :--- | :-: | :-: | :-: |
| Windows 데스크탑 (최종 제출 파일) | 0.4641 | 0.4651 | **0.3792** |
| macOS M1 (이 노트북 실행 확인, 11/13) | 0.4648 | 0.4659 | — |

- 두 환경의 차이(CV +0.0007)는 XGBoost의 CPU별 계산 차이로, 같은 환경에서는 항상 같은 결과가 나옵니다.
- 실험 과정과 근거는 `notebooks/22_ratio_features.ipynb`(비율·치환 피처), `14_model_zoo_blend.ipynb`(XGB 0.7 + LR 0.3 섞기), `17_blend_xgb_seed_avg.ipynb`(시드 평균) 참고